In [ ]:
# pip install torch xarray numpy
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
from typing import Tuple, Dict, List

device = "cuda" if torch.cuda.is_available() else "cpu"
cdtype = torch.cfloat  # complex dtype

# ----------------------------
# 1) Your dataset loader (as given)
# ----------------------------
from quam_libs.components import QuAM
from quam_libs.lib.save_utils import find_numbered_folder

def load_dataset(base_folder, target_filename = "ds", parameters=None):
    nc_files = [f for f in os.listdir(base_folder) if f.endswith('.h5')]
    is_present = target_filename in [file.split('.')[0] for file in nc_files]
    filename = [file for file in nc_files if target_filename == file.split('.')[0]][0] if is_present else None
    json_filename = "data.json"
    
    if nc_files:
        file_path = os.path.join(base_folder, filename)
        json_path = os.path.join(base_folder, json_filename)
        ds = xr.open_dataset(file_path)
        with open(json_path, 'r') as f:
            json_data = json.load(f)
        try:
            machine = QuAM.load(base_folder + "/quam_state/state.json")
        except Exception as e:
            print(f"Error loading machine: {e}")
            machine = None
        qubits = [machine.qubits[qname] for qname in ds.qubit.values] if machine is not None else list(ds.qubit.values)
        if parameters is not None:
            for param_name, param_value in parameters:
                if param_name != "load_data_id":
                    if param_name in json_data["initial_parameters"]:
                        setattr(parameters, param_name, json_data["initial_parameters"][param_name])
            return ds, machine, json_data, qubits, parameters
        else:
            return ds, machine, json_data, qubits
    else:
        print(f"No .nc file found in folder: {base_folder}")
        return None

# ----------------------------
# 2) Complex layers
# ----------------------------
class ComplexLinear(nn.Module):
    """y = x W^T + b   with complex weights/bias."""
    def __init__(self, in_features, out_features):
        super().__init__()
        w = 0.1 * (torch.randn(out_features, in_features) + 1j*torch.randn(out_features, in_features))
        b = torch.zeros(out_features, dtype=cdtype)
        self.weight = nn.Parameter(w.to(cdtype))
        self.bias   = nn.Parameter(b)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, in_features] complex
        return x @ self.weight.transpose(0,1) + self.bias

class ComplexModReLU(nn.Module):
    """
    modReLU (per-feature real bias):
      f(z) = ReLU(|z| + b) * z/|z|
    """
    def __init__(self, features: int):
        super().__init__()  # <-- the missing parentheses caused the error
        self.b = nn.Parameter(torch.zeros(features, dtype=torch.float32))

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z: [B, features] complex
        mag = torch.abs(z) + 1e-9
        gate = F.relu(mag + self.b) / mag
        return gate * z


class ComplexClassifier(nn.Module):
    """
    Complex MLP: C -> C^H -> C^H -> C^2 ; real logits via |.|^2
    """
    def __init__(self, hidden=32):
        super().__init__()
        self.l1 = ComplexLinear(1, hidden)
        self.a1 = ComplexModReLU(hidden)
        self.l2 = ComplexLinear(hidden, hidden)
        self.a2 = ComplexModReLU(hidden)
        self.out = ComplexLinear(hidden, 2)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = z.unsqueeze(-1)       # [B]c -> [B,1]c
        x = self.a1(self.l1(x))
        x = self.a2(self.l2(x))
        z_logits = self.out(x)    # [B,2] complex
        logits = torch.abs(z_logits)**2
        return logits

# ----------------------------
# 3) Dataset plumbing (xarray -> complex tensors)
# ----------------------------

# Try multiple naming schemes for your ds variables:
CANDIDATE_VAR_SETS = [
    # (ground I,Q), (excited I,Q)
    (("I_g", "Q_g"), ("I_e", "Q_e")),
    (("Ig", "Qg"), ("Ie", "Qe")),
    (("I0", "Q0"), ("I1", "Q1")),
    # Add more here if your files use other names
]

def find_var_names(ds: xr.Dataset) -> Tuple[Tuple[str,str], Tuple[str,str]]:
    names = set(ds.data_vars.keys())
    for (Ig,Qg),(Ie,Qe) in CANDIDATE_VAR_SETS:
        if {Ig,Qg}.issubset(names) and {Ie,Qe}.issubset(names):
            return (Ig,Qg),(Ie,Qe)
    raise KeyError(f"Could not find a valid (I/Q) variable set in dataset. Available: {sorted(names)}")

def get_iq_for_qubit(ds: xr.Dataset, qubit: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns two np arrays of complex64:
      z_g: ground-class complex samples
      z_e: excited-class complex samples
    """
    (Ig,Qg),(Ie,Qe) = find_var_names(ds)
    # Expect dims include 'qubit' and a sample dimension (often 'shot' or similar).
    Ig_v = ds[Ig].sel(qubit=qubit).values
    Qg_v = ds[Qg].sel(qubit=qubit).values
    Ie_v = ds[Ie].sel(qubit=qubit).values
    Qe_v = ds[Qe].sel(qubit=qubit).values

    # Flatten to 1D if needed
    Ig_v = Ig_v.reshape(-1)
    Qg_v = Qg_v.reshape(-1)
    Ie_v = Ie_v.reshape(-1)
    Qe_v = Qe_v.reshape(-1)

    z_g = Ig_v.astype(np.float32) + 1j*Qg_v.astype(np.float32)
    z_e = Ie_v.astype(np.float32) + 1j*Qe_v.astype(np.float32)
    return z_g, z_e

def complex_standardize(z: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    """
    Complex "z-score": subtract complex mean, divide by real scalar std of magnitude.
    Keeps data complex.
    """
    z_mean = z.mean()
    zc = z - z_mean
    std_mag = torch.abs(zc).float().std().clamp_min(1e-6)
    zc = zc / std_mag
    stats = {"mean": z_mean, "std_mag": std_mag}
    return zc, stats

class ComplexIQDataset(torch.utils.data.Dataset):
    def __init__(self, z: np.ndarray, y: np.ndarray):
        self.z = torch.from_numpy(z).to(cdtype)
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return self.z.shape[0]

    def __getitem__(self, idx):
        return self.z[idx], self.y[idx]

# ----------------------------
# 4) Training utilities
# ----------------------------
def train_one_qubit(
    ds: xr.Dataset,
    qubit: str,
    hidden: int = 32,
    batch_size: int = 512,
    steps: int = 3000,
    lr: float = 1e-3,
    val_ratio: float = 0.2,
    save_dir: str = "./models_complex_iq"
):
    os.makedirs(save_dir, exist_ok=True)

    # Extract complex samples per class
    z_g, z_e = get_iq_for_qubit(ds, qubit)
    y_g = np.zeros_like(z_g, dtype=np.int64)
    y_e = np.ones_like(z_e, dtype=np.int64)

    z_all = np.concatenate([z_g, z_e], axis=0)
    y_all = np.concatenate([y_g, y_e], axis=0)

    # Shuffle once
    perm = np.random.permutation(z_all.shape[0])
    z_all = z_all[perm]
    y_all = y_all[perm]

    # Torch tensors
    z_all_t = torch.from_numpy(z_all).to(cdtype).to(device)
    y_all_t = torch.from_numpy(y_all).long().to(device)

    # Standardize (complex-wise)
    z_all_t, stats = complex_standardize(z_all_t)

    # Split train/val
    n = z_all_t.shape[0]
    n_val = int(val_ratio * n)
    z_val, y_val = z_all_t[:n_val], y_all_t[:n_val]
    z_tr,  y_tr  = z_all_t[n_val:], y_all_t[n_val:]

    train_ds = torch.utils.data.TensorDataset(z_tr, y_tr)
    val_ds   = torch.utils.data.TensorDataset(z_val, y_val)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    # Model/opt
    model = ComplexClassifier(hidden=hidden).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    # Training loop
    model.train()
    best_val_acc = 0.0
    for step in range(1, steps+1):
        for z_b, y_b in train_loader:
            logits = model(z_b)
            loss = loss_fn(logits, y_b)
            opt.zero_grad()
            loss.backward()
            opt.step()

        if step % 200 == 0 or step == steps:
            # Eval
            model.eval()
            with torch.no_grad():
                correct, total = 0, 0
                for z_b, y_b in val_loader:
                    pred = model(z_b).argmax(dim=-1)
                    correct += (pred == y_b).sum().item()
                    total += y_b.numel()
                val_acc = correct / max(1,total)
            model.train()
            print(f"[{qubit}] step {step:4d} | val acc {val_acc*100:.2f}%")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                # Save best
                save_path = os.path.join(save_dir, f"model_{qubit}.pt")
                payload = {
                    "state_dict": model.state_dict(),
                    "hidden": hidden,
                    "stats": {"mean_real": stats["mean"].real.item(),
                              "mean_imag": stats["mean"].imag.item(),
                              "std_mag": float(stats["std_mag"].item())}
                }
                torch.save(payload, save_path)

    print(f"[{qubit}] best val acc: {best_val_acc*100:.2f}% | saved to {os.path.join(save_dir, f'model_{qubit}.pt')}")

def load_model_for_qubit(qubit: str, save_dir: str = "./models_complex_iq") -> Tuple[ComplexClassifier, Dict]:
    path = os.path.join(save_dir, f"model_{qubit}.pt")
    payload = torch.load(path, map_location=device)
    model = ComplexClassifier(hidden=payload["hidden"]).to(device)
    model.load_state_dict(payload["state_dict"])
    model.eval()
    return model, payload["stats"]

def standardize_with_stats(z: torch.Tensor, stats: Dict[str, float]) -> torch.Tensor:
    z_mean = torch.tensor(stats["mean_real"], dtype=torch.float32, device=device) + \
             1j*torch.tensor(stats["mean_imag"], dtype=torch.float32, device=device)
    std_mag = torch.tensor(stats["std_mag"], dtype=torch.float32, device=device)
    return (z - z_mean) / std_mag.clamp_min(1e-6)

# ----------------------------
# 5) Putting it together
# ----------------------------
if __name__ == "__main__":
    base_folder = "/Users/mohammadkashani/Documents/Qolab/qolab-start/data/Phys763_HW3b_data/data/2025-10-13/#65_07b_IQ_Blobs_162621"
    dataset = load_dataset(base_folder, target_filename="ds")
    assert dataset is not None, "Dataset not found"
    ds = dataset[0]  # xarray.Dataset

    # Choose the qubits you want to train on (q1..q6 exist per your note)
    qubits_to_train = [f"q{i}" for i in range(1, 7)]

    # Train one model per qubit
    for q in qubits_to_train:
        if q in list(ds.qubit.values):
            train_one_qubit(
                ds, q,
                hidden=32,
                batch_size=512,
                steps=3000,
                lr=1e-3,
                val_ratio=0.2,
                save_dir="./models_complex_iq",
            )
        else:
            print(f"Skipping {q}: not present in ds.qubit")

    # -------- Example inference for one qubit (q5) --------
    # Load model and stats, then predict on raw ds data
    try:
        model, stats = load_model_for_qubit("q5", "./models_complex_iq")

        # Example: take the first 5 ground samples + 5 excited samples from ds for q5
        z_g, z_e = get_iq_for_qubit(ds, "q5")
        z_example = np.concatenate([z_g[:5], z_e[:5]])
        z_example_t = torch.from_numpy(z_example).to(cdtype).to(device)
        z_example_t = standardize_with_stats(z_example_t, stats)

        with torch.no_grad():
            probs = F.softmax(model(z_example_t), dim=-1)
            preds = probs.argmax(dim=-1).cpu().numpy()

        print("Predictions for q5 (first 5 ground + 5 excited):", preds)
        # Expect roughly 0s for first 5, 1s for next 5 if the distribution separates well
    except FileNotFoundError:
        print("No saved model for q5 yet.")


[q1] step  200 | val acc 50.50%
[q1] step  400 | val acc 50.12%
[q1] step  600 | val acc 49.00%
[q1] step  800 | val acc 51.12%
[q1] step 1000 | val acc 50.00%
[q1] step 1200 | val acc 50.00%
[q1] step 1400 | val acc 52.00%
[q1] step 1600 | val acc 50.38%
[q1] step 1800 | val acc 50.50%
[q1] step 2000 | val acc 50.50%
[q1] step 2200 | val acc 49.62%
[q1] step 2400 | val acc 50.25%
[q1] step 2600 | val acc 49.25%
[q1] step 2800 | val acc 50.75%
[q1] step 3000 | val acc 48.50%
[q1] best val acc: 52.00% | saved to ./models_complex_iq/model_q1.pt
[q2] step  200 | val acc 91.00%
[q2] step  400 | val acc 91.25%
[q2] step  600 | val acc 91.12%
[q2] step  800 | val acc 91.62%
[q2] step 1000 | val acc 91.25%
[q2] step 1200 | val acc 91.38%
[q2] step 1400 | val acc 91.50%
[q2] step 1600 | val acc 91.25%
[q2] step 1800 | val acc 91.25%
[q2] step 2000 | val acc 91.12%
[q2] step 2200 | val acc 90.88%
[q2] step 2400 | val acc 90.25%
[q2] step 2600 | val acc 90.50%
[q2] step 2800 | val acc 90.38%
[q2